In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/combined_2024_bike_data.csv')

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1233396 entries, 0 to 1233395
Data columns (total 11 columns):
 #   Column   Non-Null Count    Dtype  
---  ------   --------------    -----  
 0   대여일자     1233396 non-null  int64  
 1   대여소번호    1233396 non-null  int64  
 2   대여소명     1233396 non-null  str    
 3   대여구분코드   1233396 non-null  str    
 4   성별       843095 non-null   str    
 5   연령대코드    1233396 non-null  str    
 6   이용건수     1233396 non-null  int64  
 7   운동량      1232646 non-null  float64
 8   탄소량      1232646 non-null  float64
 9   이동거리(M)  1233396 non-null  float64
 10  이용시간(분)  1233396 non-null  int64  
dtypes: float64(3), int64(4), str(4)
memory usage: 159.2 MB


In [3]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_ratio(%)": (df.isnull().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)

display(missing_summary[missing_summary["missing_count"] > 0])

,missing_count,missing_ratio(%)
성별,390301,31.64
운동량,750,0.06
탄소량,750,0.06


In [4]:
# 결측치 처리 : fillna()
# 1. 성별 - 중앙값(median)
# df['성별'] = df['성별'].fillna( df['성별'].median() )

# # 2. Embarked - 최빈값(mode)
# df['Embarked'] = df['Embarked'].fillna( df['Embarked'].mode()[0] )
# 1. 성별 빈도수 계산 후 내림차순 정렬 (.sort_values 사용)

# 1. '성별' 컬럼의 결측치 개수 파악
null_count = df['성별'].isnull().sum()
print(f"성별 결측치 개수: {null_count}개")

# 2. 결측치 개수만큼 '남'과 '여'를 50:50 확률로 생성
# (데이터에 맞춰 '남'/'여' 대신 'M'/'F'로 변경하셔도 됩니다)
random_genders = np.random.choice(['M', 'F'], size=null_count, p=[0.5, 0.5])

# 3. 결측치 위치에 생성한 랜덤 성별 집어넣기
df.loc[df['성별'].isnull(), '성별'] = random_genders

# 4. 결과 확인 (남녀 비율이 비슷해졌는지 확인)
df['성별'].value_counts()

성별 결측치 개수: 390301개


성별
M    641127
F    592079
m       120
f        70
Name: count, dtype: int64

In [5]:
# 성별 컬럼의 소문자를 모두 대문자로 변경 (공백 제거 포함)
df['성별'] = df['성별'].str.strip().str.upper()

# 결과 확인
gender_counts  = df['성별'].value_counts()
print(gender_counts.head(10))

성별
M    641247
F    592149
Name: count, dtype: int64


In [6]:
# 2. 운동량, 탄소량- 최빈값(mode)
df['운동량'] = df['운동량'].fillna( df['운동량'].mode()[0] )
df['탄소량'] = df['탄소량'].fillna( df['탄소량'].mode()[0] )

In [7]:
# 1. '대' 문자열을 지우고 숫자만 남기기
df['연령대코드'] = df['연령대코드'].str.replace('대', '', regex=False)

# 2. 숫자로 강제 변환 (결측치나 '미지정' 글자가 섞여있다면 에러 방지를 위해 errors='coerce' 사용)
df['연령대코드'] = pd.to_numeric(df['연령대코드'], errors='coerce')

# 3. 변환 결과 확인
df['연령대코드'].value_counts()

연령대코드
20.0    194174
30.0    190885
40.0    181443
50.0    164137
60.0    118159
Name: count, dtype: int64

In [8]:
df.head(10)

,대여일자,대여소번호,대여소명,대여구분코드,성별,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분)
0,202401,102,102. 망원역 1번출구 앞,일일권,F,20.0,30,1562.85,14.77,63766.65,547
1,202401,102,102. 망원역 1번출구 앞,일일권,F,30.0,21,1082.06,9.53,41075.31,393
2,202401,102,102. 망원역 1번출구 앞,일일권,F,40.0,4,82.63,0.73,3178.67,66
3,202401,102,102. 망원역 1번출구 앞,일일권,M,50.0,3,160.96,1.63,7009.09,44
4,202401,102,102. 망원역 1번출구 앞,일일권,M,NaN,4,68.96,0.78,3375.82,40
5,202401,102,102. 망원역 1번출구 앞,일일권,F,20.0,30,1815.28,17.59,75906.92,855
6,202401,102,102. 망원역 1번출구 앞,일일권,F,30.0,15,1316.30,12.84,55325.88,517
7,202401,102,102. 망원역 1번출구 앞,일일권,F,40.0,13,833.81,7.59,32756.10,260
8,202401,102,102. 망원역 1번출구 앞,일일권,F,50.0,1,22.55,0.20,849.75,42
9,202401,102,102. 망원역 1번출구 앞,일일권,F,NaN,1,30.36,0.27,1179.61,18


In [9]:
df['연령대코드'].isnull().sum()

np.int64(384598)

In [10]:
# 1. '연령대코드'의 현재 결측치 개수 파악
age_null_count = df['연령대코드'].isnull().sum()
print(f"연령대코드 결측치 개수: {age_null_count}개")

# 결측치가 있을 때만 아래 로직을 실행합니다.
if age_null_count > 0:
    # 2. 기존 데이터의 연령대별 실제 비율(확률) 계산 (결측치 제외)
    # normalize=True를 주면 빈도수가 아닌 비율(합이 1)로 반환됩니다.
    age_probs = df['연령대코드'].value_counts(normalize=True)
    
    # 3. 계산된 비율 그대로 결측치 개수만큼 랜덤 생성
    # age_probs.index는 [20, 30, 40, 50, 60] 등의 고유값, age_probs.values는 각 확률입니다.
    random_ages = np.random.choice(
        age_probs.index, 
        size=age_null_count, 
        p=age_probs.values
    )

    # random_ages = np.random.choice([20, 30, 40, 50, 60], size=age_null_count)
    
    # 4. 결측치 위치에 생성한 랜덤 연령대 집어넣기
    df.loc[df['연령대코드'].isnull(), '연령대코드'] = random_ages
    print("결측치 채우기 완료!")

# 5. 최종 결과 및 결측치 남았는지 확인
print(df['연령대코드'].value_counts(dropna=False))

연령대코드 결측치 개수: 384598개
결측치 채우기 완료!
연령대코드
20.0    282532
30.0    276932
40.0    263468
50.0    238477
60.0    171987
Name: count, dtype: int64


In [47]:
df.describe()

,대여일자,대여소번호,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분)
count,1.233396e+06,1.233396e+06,1.233396e+06,1.233396e+06,1.233396e+06,1.233396e+06,1.233396e+06,1.233396e+06
mean,2.024066e+05,2.352501e+03,3.790167e+01,3.555189e+01,2.227954e+03,1.937286e+01,8.384110e+04,7.438958e+02
std,3.373785e+00,1.594367e+03,1.358729e+01,6.730375e+01,4.340693e+03,3.722160e+01,1.610801e+05,1.390664e+03
min,2.024010e+05,1.020000e+02,2.000000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.024040e+05,9.920000e+02,3.000000e+01,4.000000e+00,2.225600e+02,2.030000e+00,8.780000e+03,8.100000e+01
50%,2.024070e+05,2.037000e+03,4.000000e+01,1.200000e+01,7.836300e+02,7.040000e+00,3.048078e+04,2.740000e+02
75%,2.024090e+05,3.793000e+03,5.000000e+01,3.800000e+01,2.422977e+03,2.141000e+01,9.265555e+04,8.240000e+02
max,2.024120e+05,6.178000e+03,6.000000e+01,2.593000e+03,2.877765e+05,2.825580e+03,1.222876e+07,1.099940e+05


In [14]:
df.head(10)

,대여일자,대여소번호,대여소명,대여구분코드,성별,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분)
0,202401,102,102. 망원역 1번출구 앞,일일권,F,20.0,30,1562.85,14.77,63766.65,547
1,202401,102,102. 망원역 1번출구 앞,일일권,F,30.0,21,1082.06,9.53,41075.31,393
2,202401,102,102. 망원역 1번출구 앞,일일권,F,40.0,4,82.63,0.73,3178.67,66
3,202401,102,102. 망원역 1번출구 앞,일일권,M,50.0,3,160.96,1.63,7009.09,44
4,202401,102,102. 망원역 1번출구 앞,일일권,M,20.0,4,68.96,0.78,3375.82,40
5,202401,102,102. 망원역 1번출구 앞,일일권,F,20.0,30,1815.28,17.59,75906.92,855
6,202401,102,102. 망원역 1번출구 앞,일일권,F,30.0,15,1316.30,12.84,55325.88,517
7,202401,102,102. 망원역 1번출구 앞,일일권,F,40.0,13,833.81,7.59,32756.10,260
8,202401,102,102. 망원역 1번출구 앞,일일권,F,50.0,1,22.55,0.20,849.75,42
9,202401,102,102. 망원역 1번출구 앞,일일권,F,20.0,1,30.36,0.27,1179.61,18


In [12]:
# 1. 0값 비율 확인
zero_distance = (df['이동거리(M)'] == 0).sum()
zero_time = (df['이용시간(분)'] == 0).sum()
total_rows = len(df)

print(f"전체 데이터 수: {total_rows:,}건")
print(f"이동거리 0건: {zero_distance:,}건 ({zero_distance/total_rows*100:.2f}%)")
print(f"이용시간 0건: {zero_time:,}건 ({zero_time/total_rows*100:.2f}%)")

전체 데이터 수: 1,233,396건
이동거리 0건: 2,382건 (0.19%)
이용시간 0건: 123건 (0.01%)


In [15]:
# 2. 유효한 데이터만 필터링 (2분 이상 & 10m 이상)
df_clean = df[(df['이용시간(분)'] >= 2) & (df['이동거리(M)'] >= 10)].copy()

print(f"필터링 후 남은 데이터 수: {len(df_clean):,}건")

df_clean['대여일자']

df_clean.to_parquet('combined_2024_bike_clean.parquet', engine='fastparquet', index=False)


필터링 후 남은 데이터 수: 1,230,384건


In [74]:
# 3. 정제된 데이터를 가지고 월별 집계(groupby) 진행
monthly_data = df_clean.groupby('대여일자').agg({
    '이용건수': 'sum',
    '이동거리(M)': 'mean',  # 0이 빠졌으므로 훨씬 정확한 월별 평균 이동거리가 계산됨
    '이용시간(분)': 'mean',
    '운동량': 'mean'
}).reset_index()

monthly_data

,대여일자,이용건수,이동거리(M),이용시간(분),운동량
0,202401,1983097,42200.773597,395.921554,1134.701127
1,202402,2031459,45915.671126,417.771622,1228.746098
2,202403,3151034,72895.104829,639.789336,1936.897576
3,202404,4599767,113545.972218,969.670805,3000.678808
4,202405,4788873,116053.494751,986.858963,3073.313380
5,202406,4857815,113672.547184,971.391396,3019.835079
6,202407,3723042,80308.731535,713.813675,2141.887871
7,202408,3998034,82824.244625,777.902804,2206.968600
8,202409,4250588,99225.233143,872.489549,2624.570723
9,202410,4686560,104361.532965,938.693620,2762.654578
